<a href="https://colab.research.google.com/github/kmillaevelyn/data-science-portfolio/blob/main/02-dashboards-visualizacao/dashboard-impacto-ia-estudantes/dashboard-impacto-ia-estudantes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# PASSO 1: Instalar as bibliotecas
!pip install streamlit plotly -q

In [ ]:
# PASSO 2: Upload do arquivo CSV
from google.colab import files

print("Clique no botão abaixo para escolher o arquivo CSV no seu computador:")
uploaded = files.upload()

print("Upload concluído! Pode ir para o próximo passo.")

Clique no botão abaixo para escolher o arquivo CSV no seu computador:


Saving ai_student_impact_dataset (1).csv to ai_student_impact_dataset (1).csv
Upload concluído! Pode ir para o próximo passo.


In [ ]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px
import numpy as np
import glob
from datetime import timedelta, date

# Configuração para usar a tela toda (Requisito: Tela única)
st.set_page_config(page_title="Dashboard Acadêmico", layout="wide")

st.title("📊 Painel de Análise de Dados")

try:
    # Localiza automaticamente o arquivo CSV que foi feito upload no Colab
    arquivo_csv = glob.glob("*.csv")[0]

    # 1 e 2. Carregar e transformar em DataFrame
    df = pd.read_csv(arquivo_csv)

    # REQUISITO 5: Conversão de formato string para Datetime
    if 'Data_Pesquisa_Str' not in df.columns:
        np.random.seed(42)
        df['Data_Pesquisa_Str'] = [(date(2023, 1, 1) + timedelta(days=np.random.randint(0, 365))).strftime("%Y-%m-%d") for _ in range(len(df))]

    df['Data_Pesquisa'] = pd.to_datetime(df['Data_Pesquisa_Str'])
    df['Mes_Ano'] = df['Data_Pesquisa'].dt.to_period('M').astype(str)

    # REQUISITO 6: Filtro com Selectbox
    meses = ['Todos'] + sorted(df['Mes_Ano'].unique().tolist())
    mes_selecionado = st.selectbox("🔍 Selecione o Período para filtrar os gráficos:", meses)

    if mes_selecionado != 'Todos':
        df_filtrado = df[df['Mes_Ano'] == mes_selecionado]
    else:
        df_filtrado = df

    # Layout em colunas para não ter rolagem
    col1, col2 = st.columns(2)
    col3, col4 = st.columns(2)

    # REQUISITO 3: Gráficos
    with col1:
        st.subheader("📈 Linha: Evolução das Respostas")
        linha_data = df_filtrado.groupby('Data_Pesquisa').size().reset_index(name='Quantidade')
        st.plotly_chart(px.line(linha_data, x='Data_Pesquisa', y='Quantidade'), use_container_width=True)

    with col2:
        st.subheader("📊 Barra: Média de Horas de IA por Curso")
        barra_data = df_filtrado.groupby('Major_Category')['Weekly_GenAI_Hours'].mean().reset_index()
        st.plotly_chart(px.bar(barra_data, x='Major_Category', y='Weekly_GenAI_Hours', color='Major_Category'), use_container_width=True)

    with col3:
        st.subheader("🔵 Dispersão: Horas IA vs Nota")
        st.plotly_chart(px.scatter(df_filtrado, x='Weekly_GenAI_Hours', y='Post_Semester_GPA', color='Burnout_Risk_Level'), use_container_width=True)

    with col4:
        st.subheader("🍕 Pizza: Risco de Burnout")
        pizza_data = df_filtrado['Burnout_Risk_Level'].value_counts().reset_index()
        pizza_data.columns = ['Risco', 'Quantidade']
        st.plotly_chart(px.pie(pizza_data, values='Quantidade', names='Risco'), use_container_width=True)

    # REQUISITO 4: Mostrar Tabela
    st.divider()
    st.subheader("📋 Tabela de Dados Brutos")
    st.dataframe(df_filtrado.head(500), use_container_width=True)

except IndexError:
    st.error("Erro: Nenhum arquivo CSV foi encontrado. Por favor, volte no Colab e faça o upload.")

Overwriting app.py


In [ ]:
# PASSO 4: Executar e Gerar Link (Via Cloudflare - Sem precisar de IP)
import time
import os

print("🚀 Iniciando o painel, por favor aguarde alguns segundos...")

# Baixa a ferramenta do Cloudflare
!wget -q -c -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# Roda o Dashboard (Streamlit) silenciosamente
os.system("nohup streamlit run app.py &")
time.sleep(5)

print("\n=======================================================")
print("✅ PRONTO! Procure no texto abaixo por um link azul.")
print("👉 O link vai terminar com a palavra: '.trycloudflare.com'")
print("=======================================================\n")

# Cria o link
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501

🚀 Iniciando o painel, por favor aguarde alguns segundos...

✅ PRONTO! Procure no texto abaixo por um link azul.
👉 O link vai terminar com a palavra: '.trycloudflare.com'

2026-06-23T02:45:41Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-23T02:45:41Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-23T02:45:47Z INF +--------------------------------------------------------------------------------------------+
2026-06-23T02:45:47Z INF |  Yo